<a href="https://colab.research.google.com/github/NethmiKaveeshaE23174/Statistical-Learning-e23174/blob/main/Assignment7_C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q. Bayesian Estimation of a User Ability Parameter from Item Responses

An online learning platform presents a user with a sequence of $n$ multiple-choice questions **one at a time**. Each question is either answered correctly or incorrectly, allowing the platform to update its estimate of the user's ability dynamically after every response.

Let $Y_i$ denote the user's response to the $i$-th item encountered:

$$Y_i=
\begin{cases}
1, & \text{if the user answers item } i \text{ correctly},\\
0, & \text{if the user answers item } i \text{ incorrectly}.
\end{cases}$$

The platform assumes that the probability of a correct response is governed by a two-parameter logistic (2PL) item response model. Specifically, conditional on the user's latent ability parameter $\Theta=\theta$, the response probability for item $i$ is:

$$P(Y_i=1\mid \Theta=\theta)=p_i(\theta)=\frac{1}{1+e^{-a_i(\theta-b_i)}},$$

where $a_i>0$ is the known discrimination parameter, and $b_i$ is the known difficulty parameter of item $i$.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed responses** up to the current step $k$ (where $1 \le k \le n$).

Before observing any responses, the platform initializes the user's latent ability estimate with a standard normal prior distribution:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta^2}{2}\right) \quad \text{implying} \quad \Theta \sim \mathscr{N}(0,1).$$

As the user progresses, the posterior distribution at step $k-1$ serves as the prior distribution for step $k$.

---

### Tasks

1. **Visualizing the Mechanics:** Plot $P(Y_i=1\mid \Theta=\theta)$ vs $\theta$ using Plotly for two distinct values of $a_i$, where one of those $a_i$ values is paired with three different difficulty values of $b_i$. Interpret how moving $b_i$ shifts the curve horizontally.
2. **Sequential Likelihood Contribution:** Write down the likelihood contribution $L(y_k \mid \theta)$ of a *single* new response $y_k$ at step $k$, given the latent ability $\theta$. Then, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.
3. **Mathematical Formulation of the Running Update:** Write down the recursive relationship for the posterior density at step $k$, denoted $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$, up to a proportionality constant, using the prior state $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ and the new observation $y_k$.
4. **Dynamic Shifting:** Explain how a correct answer ($y_k = 1$) to a highly difficult item (large $b_k$) mathematically shifts the peak of the running posterior density distribution relative to the previous step.
5. **Tracking Certainty and Sharpness:** Explain how the discrimination parameter $a_k$ of the current item alters the variance (or "sharpness") of the distribution during a running update. What happens when $a_k$ is very large versus very small?
6. **Numerical Implementation of a Running Grid:** Describe a algorithmic approach to numerically approximate and maintain this running posterior density function on a fixed grid of $\theta$-values. Explicitly state how you would perform the sequential normalization step computationally after an item is answered.


7. **Evaluating Convergence over the Timeline:** Suppose the user's true, hidden latent ability is $\theta_{\text{true}} = 0.75$. Write a Python script that extends your previous grid simulation to track the performance of the running estimators over a sequence of $n = 20$ items.
* **Simulate Responses:** Dynamically generate the user's responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against the true response probability $p_k(\theta_{\text{true}})$. Give each item a random difficulty $b_k \sim \mathscr{N}(0, 1)$ and a random discrimination $a_k \sim \text{Uniform}(0.5, 2.0)$.
* **Track Estimators:** At each step $k$, calculate and store the running Posterior Mean ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$) and the running Maximum A Posteriori ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$) estimate.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $20$. Add a static horizontal reference line at $y = 0.75$ representing $\theta_{\text{true}}$.
* **Analysis:** Briefly explain how the distance between your estimators and $\theta_{\text{true}}$ changes as $k$ increases, and interpret what this implies about the platform's confidence in its measurement.


##  Answer

# Bayesian Estimation of User Ability ($\theta$) from Item Responses (2PL IRT Model)

---

## 1. Visualizing the Mechanics of the 2PL IRT Model

The Item Response Function (IRF) under the 2PL model is given by:

$$P(Y_i = 1 \mid \Theta = \theta) = p_i(\theta) = \frac{1}{1 + e^{-a_i(\theta - b_i)}}$$

### Horizontal Shift Interpretation ($b_i$):
* **Difficulty Parameter ($b_i$):** Represents the location on the ability scale ($\theta$) where the probability of answering correctly is exactly $0.5$ (since $p_i(b_i) = \frac{1}{1 + e^0} = 0.5$).
* Increasing $b_i$ shifts the curve **rightward** (requires higher ability to pass).
* Decreasing $b_i$ shifts the curve **leftward** (easier item, lower ability yields higher probability).

In [ ]:
import numpy as np
import plotly.graph_objects as go

# Define the 2PL Item Response Function
def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

# Latent ability grid
theta_vals = np.linspace(-6, 6, 300)

# Configurations: 2 distinct 'a' values, one paired with 3 distinct 'b' values
curves = [
    {"a": 0.5, "b": 0, "dash": "dash"},
    {"a": 1.5, "b": -2, "dash": "solid"},
    {"a": 1.5, "b": 0, "dash": "solid"},
    {"a": 1.5, "b": 2, "dash": "solid"},
]

fig1 = go.Figure()

for c in curves:
    a, b, dash = c["a"], c["b"], c["dash"]
    fig1.add_trace(go.Scatter(
        x=theta_vals,
        y=p_i(theta_vals, a, b),
        mode='lines',
        name=f"a = {a}, b = {b}",
        line=dict(dash=dash, width=2.5)
    ))

fig1.update_layout(
    title="Two-Parameter Logistic (2PL) Item Response Curves",
    xaxis_title="Latent Ability (θ)",
    yaxis_title="P(Y_i = 1 | θ)",
    template="plotly_white",
    hovermode="x unified"
)

fig1.show()

---

## 2. Sequential Likelihood Contribution & Joint History

### Single Response Likelihood Contribution at Step $k$:
$$L(y_k \mid \theta) = [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k}$$

### Joint Likelihood Function for Running Vector $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$:
Assuming conditional independence given $\Theta = \theta$:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^{k} [p_i(\theta)]^{y_i} [1 - p_i(\theta)]^{1 - y_i}$$

---

## 3. Mathematical Formulation of the Running Update

Using Bayes' Theorem recursively:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

Explicitly written:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k} \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

---

## 4. Dynamic Shifting Mechanics

When $y_k = 1$ for a difficult item (large $b_k$):
* $p_k(\theta)$ is near $0$ for low/moderate $\theta$ and climbs sharply near/after $b_k$.
* Multiplying the prior density $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ by this monotonically increasing sigmoid suppresses lower values of $\theta$ and magnifies higher values of $\theta$.
* As a result, the posterior mode (peak) **shifts sharply to the right** towards higher ability values.

---

## 5. Tracking Certainty and Sharpness ($a_k$)

* **High Discrimination ($a_k \gg 0$):** $p_k(\theta)$ exhibits a steep step-like transition at $b_k$. The derivative of the log-likelihood is large, imparting strong Fisher information. This significantly reduces posterior variance, yielding a sharper peak.
* **Low Discrimination ($a_k \approx 0$):** $p_k(\theta)$ is relatively flat around $0.5$. The likelihood factor carries almost no information about $\theta$, leaving the posterior almost unchanged and maintaining high variance.

---

## 6. Numerical Grid Implementation & Computational Normalization

1. **Define Grid:** Construct an evenly spaced grid $\boldsymbol{\theta} = [\theta_1, \theta_2, \dots, \theta_M]$ over $[-5, 5]$.
2. **Initialization:** Initialize prior density vector $\mathbf{f}^{(0)} = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\boldsymbol{\theta}^2}{2}\right)$.
3. **Sequential Step:**
   * Compute likelihood array $\mathbf{L}_k = [p_k(\boldsymbol{\theta})]^{\mathbf{y}_k} [1 - p_k(\boldsymbol{\theta})]^{1 - \mathbf{y}_k}$.
   * Compute unnormalized posterior: $\mathbf{f}_{\text{unnorm}}^{(k)} = \mathbf{L}_k \odot \mathbf{f}^{(k-1)}$.
   * **Computational Normalization Step:** Evaluate normalization constant $Z_k$ using the trapezoidal numerical integration rule:
     $$Z_k = \int_{\min(\boldsymbol{\theta})}^{\max(\boldsymbol{\theta})} f_{\text{unnorm}}^{(k)}(\theta) \, d\theta \approx \text{trapz}(\mathbf{f}_{\text{unnorm}}^{(k)}, \boldsymbol{\theta})$$
   * Update normalized posterior: $\mathbf{f}^{(k)} = \frac{\mathbf{f}_{\text{unnorm}}^{(k)}}{Z_k}$.

---

## 7. Evaluating Convergence over Timeline Simulation

In [ ]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# Set seed for reproducibility
np.random.seed(42)

# Parameters
theta_true = 0.75
n_items = 20
theta_grid = np.linspace(-5, 5, 1000)

# Random item parameters
a_params = np.random.uniform(0.5, 2.0, size=n_items)
b_params = np.random.normal(0, 1, size=n_items)

# 2PL function
def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

# Tracking arrays
running_bayes = [0.0]  # Prior mean N(0,1) is 0
running_map = [0.0]    # Prior mode N(0,1) is 0
steps = list(range(n_items + 1))

# Initial Prior: N(0, 1)
current_posterior = stats.norm.pdf(theta_grid, 0, 1)

# Simulation Loop
for k in range(n_items):
    a_k = a_params[k]
    b_k = b_params[k]

    # Simulate response using true theta
    p_true = p_i(theta_true, a_k, b_k)
    y_k = 1 if np.random.uniform(0, 1) < p_true else 0

    # Likelihood over grid
    p_grid = p_i(theta_grid, a_k, b_k)
    likelihood = (p_grid ** y_k) * ((1 - p_grid) ** (1 - y_k))

    # Posterior update and trapezoidal normalization
    current_posterior *= likelihood
    Z_k = np.trapezoid(current_posterior, theta_grid)
    current_posterior /= Z_k

    # Estimates
    bayes_est = np.trapezoid(theta_grid * current_posterior, theta_grid)
    map_est = theta_grid[np.argmax(current_posterior)]

    running_bayes.append(bayes_est)
    running_map.append(map_est)

# Plotting with Plotly
fig2 = go.Figure()

# True ability reference line
fig2.add_hline(
    y=theta_true,
    line_dash="dash",
    line_color="red",
    annotation_text=f"True Ability (θ = {theta_true})"
)

# Posterior Mean Line
fig2.add_trace(go.Scatter(
    x=steps, y=running_bayes,
    mode='lines+markers',
    name='Posterior Mean (θ̂_Bayes)',
    line=dict(color='blue', width=2.5)
))

# MAP Line
fig2.add_trace(go.Scatter(
    x=steps, y=running_map,
    mode='lines+markers',
    name='MAP Estimate (θ̂_MAP)',
    line=dict(color='green', width=2)
))

fig2.update_layout(
    title="Convergence of Latent Ability Estimators (θ) Over Time",
    xaxis_title="Item Sequence (k)",
    yaxis_title="Estimated Ability (θ̂)",
    template="plotly_white",
    hovermode="x unified"
)

fig2.show()

### Analysis & Interpretation

* **Convergence Trajectory:** As $k$ increases, both $\widehat{\theta}_{\text{Bayes}}^{(k)}$ and $\widehat{\theta}_{\text{MAP}}^{(k)}$ stabilize and narrow down around $\theta_{\text{true}} = 0.75$.
* **Confidence & Uncertainty Reduction:** Accumulated response data dominates the initial prior standard normal belief $\mathcal{N}(0, 1)$. The posterior variance contracts with each item (proportional to accumulated Fisher information), demonstrating that the platform's measurement confidence steadily increases over the timeline.

# Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

An e-commerce platform wants to optimize its recommendation engine by dynamically estimating the click-through rate (CTR) of a newly launched advertisement. Since user traffic arrives continuously, the platform updates its belief about the advertisement's performance **one impression at a time** rather than waiting for large batch updates.

Let $\Theta = \theta$ represent the true, hidden conversion rate (probability of a click) of the advertisement, where $\theta \in [0, 1]$.

Let $Y_k$ denote a single user's interaction with the advertisement at time step $k$:

$$Y_k =
\begin{cases}
1, & \text{if the user clicks the advertisement}, \\
0, & \text{if the user does not click the advertisement}.
\end{cases}$$

The platform assumes that conditional on the true conversion rate $\Theta = \theta$, each user interaction is an independent Bernoulli trial:

$$P(Y_k = 1 \mid \Theta = \theta) = \theta$$

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed user interactions** up to the current impression step $k$ (where $1 \le k \le n$).

Before observing any data, the platform assigns a flexible **Beta distribution** as the initial prior over the unknown parameter $\Theta$:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\mathrm{B}(\alpha_0, \beta_0)} \theta^{\alpha_0 - 1} (1 - \theta)^{\beta_0 - 1} \quad \text{implying} \quad \Theta \sim \text{Beta}(\alpha_0, \beta_0)$$

where $\mathrm{B}(\cdot, \cdot)$ is the Beta function acting as the normalizing constant. Under a sequential framework, the posterior distribution at step $k-1$ serves directly as the prior distribution for step $k$.

---

**Tasks**

**1. Structural Probability and Properties**
Plot the probability density function (PDF) of a $\text{Beta}(\alpha, \beta)$ distribution using Plotly for three distinct parameter pairs:

* Uninformative state: $(\alpha=1, \beta=1)$
* Right-skewed state: $(\alpha=2, \beta=8)$
* Left-skewed state: $(\alpha=8, \beta=2)$

Interpret how changing the balance between $\alpha$ and $\beta$ shifts the center of mass of the density function over the domain $[0, 1]$.

**2. Sequential Likelihood and Joint History**

Write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* isolated response $y_k$ at step $k$, given the click probability $\theta$. Following this, express the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

**3. Closed-Form Analytical Updates (Conjugacy)**

Using Bayes' Theorem, derive the recursive algebraic relationship for the posterior density at step $k$, denoted as $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$. Prove analytically that the posterior remains in the Beta family (**Beta-Binomial Conjugacy**) by explicitly writing down the closed-form update parameters $\alpha_k$ and $\beta_k$ as simple arithmetic updates of $\alpha_{k-1}$, $\beta_{k-1}$, and $y_k$. Also compute the **Posterior Mean** of the latent parameter $\Theta$ at time step $k$ (i.e. $\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}]$).


**4. Dynamic Shifting Mechanics**

Explain how an observed click ($y_k = 1$) vs. a non-click ($y_k = 0$) shifts the peak of the running density distribution mathematically. Contrast this analytical framework against non-conjugate setups (such as the 2PL IRT model) where numerical grid integration is strictly required.

**5. Running Point Estimators**

State the exact closed-form equations used to evaluate the following point estimates at step $k$ directly from the updated shape parameters $\alpha_k$ and $\beta_k$:

* **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

**6. Performance Tracking and Convergence Analysis**

Suppose the advertisement's true, hidden click-through rate is $\theta_{\text{true}} = 0.35$. Write a Python script to track the performance of your closed-form sequential estimators over a timeline of $n = 100$ impressions:

* **Initialize State:** Set the base prior parameters to $\alpha_0 = 1, \beta_0 = 1$ (representing uniform initial uncertainty).
* **Simulate Responses:** Dynamically generate user responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against $\theta_{\text{true}}$.
* **Track Estimators:** Loop through each step, update $\alpha_k$ and $\beta_k$ analytically, and store the computed values for $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $100$. Add a static horizontal reference line at $y = 0.35$ representing $\theta_{\text{true}}$.
* **Analysis:** Explain how the distance between your estimators and $\theta_{\text{true}}$ responds as the sampling size $k$ approaches $100$. What does this imply about the accumulation of evidence over time relative to the choice of the initial prior?

**Answer**

# Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

---

## 1. Structural Probability and Properties

The PDF of a $\text{Beta}(\alpha, \beta)$ distribution for $\theta \in [0, 1]$ is defined as:

$$f(\theta; \alpha, \beta) = \frac{1}{\mathrm{B}(\alpha, \beta)} \theta^{\alpha - 1} (1 - \theta)^{\beta - 1}$$

### Center of Mass & Skewness Interpretation:
* **Uninformative State ($\alpha=1, \beta=1$):** Reduces to a Uniform distribution over $[0, 1]$, where every rate is equally probable.
* **Right-Skewed State ($\alpha=2, \beta=8$):** $\beta > \alpha$ pulls the density's center of mass towards $0$ (low CTR expectation).
* **Left-Skewed State ($\alpha=8, \beta=2$):** $\alpha > \beta$ pushes the density's center of mass towards $1$ (high CTR expectation).
* **General Rule:** The expected value $\mathbb{E}[\Theta] = \frac{\alpha}{\alpha + \beta}$ dictates the center of mass.

In [ ]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# Theta domain
theta_vals = np.linspace(0, 1, 500)

# Parameter pairs
beta_configs = [
    {"alpha": 1, "beta": 1, "name": "Uninformative: Beta(1, 1)"},
    {"alpha": 2, "beta": 8, "name": "Right-Skewed: Beta(2, 8)"},
    {"alpha": 8, "beta": 2, "name": "Left-Skewed: Beta(8, 2)"}
]

fig1 = go.Figure()

for cfg in beta_configs:
    a, b, name = cfg["alpha"], cfg["beta"], cfg["name"]
    pdf_vals = stats.beta.pdf(theta_vals, a, b)
    fig1.add_trace(go.Scatter(
        x=theta_vals,
        y=pdf_vals,
        mode='lines',
        name=name,
        line=dict(width=2.5)
    ))

fig1.update_layout(
    title="Beta Distribution Densities for Different Shape Parameters",
    xaxis_title="Click-Through Rate (θ)",
    yaxis_title="Probability Density f(θ)",
    template="plotly_white",
    hovermode="x unified"
)

fig1.show()

---

## 2. Sequential Likelihood and Joint History

### Single Response Likelihood Contribution at Step $k$:
$$L(y_k \mid \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$

### Joint Likelihood Function for Running Vector $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$:
Assuming independent Bernoulli trials given $\Theta = \theta$:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^{k} \theta^{y_i} (1 - \theta)^{1 - y_i} = \theta^{\sum_{i=1}^k y_i} (1 - \theta)^{k - \sum_{i=1}^k y_i}$$

---

## 3. Closed-Form Analytical Updates (Conjugacy)

Using Bayes' Theorem recursively:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

Substituting $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) \propto \theta^{\alpha_{k-1}-1} (1-\theta)^{\beta_{k-1}-1}$:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \left[\theta^{y_k} (1 - \theta)^{1 - y_k}\right] \cdot \left[\theta^{\alpha_{k-1}-1} (1 - \theta)^{\beta_{k-1}-1}\right]$$

$$\implies f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \theta^{(\alpha_{k-1} + y_k) - 1} (1 - \theta)^{(\beta_{k-1} + 1 - y_k) - 1}$$

### Analytical Proof of Beta-Binomial Conjugacy:
The posterior density matches the functional form of a Beta distribution, proving conjugacy. The exact arithmetic parameter updates are:

$$\alpha_k = \alpha_{k-1} + y_k$$

$$\beta_k = \beta_{k-1} + (1 - y_k)$$

### Posterior Mean at Step $k$:
$$\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)} = \mathbf{y}^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k} = \frac{\alpha_0 + \sum_{i=1}^k y_i}{\alpha_0 + \beta_0 + k}$$

---

## 4. Dynamic Shifting Mechanics

* **Observed Click ($y_k = 1$):** Increments $\alpha_k$ by $1$ while $\beta_k$ remains unchanged. This multiplies the density function by $\theta$, weighting higher values of $\theta$ more heavily and shifting the mode rightward.
* **Observed Non-click ($y_k = 0$):** Increments $\beta_k$ by $1$ while $\alpha_k$ remains unchanged. This multiplies the density function by $(1-\theta)$, weighting lower values of $\theta$ more heavily and shifting the mode leftward.
* **Analytical vs. Non-Conjugate Frameworks:**
  * Under **Conjugacy** (Beta-Binomial), the posterior stays in a known parameter family, enabling **exact $O(1)$ arithmetic updates** without integration.
  * In **Non-Conjugate models** (like 2PL IRT), the likelihood function cannot be combined with the prior into a standard distribution, forcing numerical integration over a discretized grid.

---

## 5. Running Point Estimators

For an updated state $\text{Beta}(\alpha_k, \beta_k)$:

1. **Running Posterior Mean ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$):**
$$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \frac{\alpha_k}{\alpha_k + \beta_k}$$

2. **Running Maximum A Posteriori ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$) [for $\alpha_k, \beta_k > 1$]:**
$$\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2}$$

In [ ]:
import numpy as np
import plotly.graph_objects as go

# Set seed for reproducibility
np.random.seed(42)

# Parameters
theta_true = 0.35
n_impressions = 100

# Base Prior: Uniform Beta(1, 1)
alpha_k = 1.0
beta_k = 1.0

# Tracking arrays
running_bayes = [alpha_k / (alpha_k + beta_k)]
# Initial MAP for Beta(1,1) is undefined/flat, set to 0.5 (or prior mode)
running_map = [0.5]
steps = list(range(n_impressions + 1))

# Simulation Loop
for k in range(1, n_impressions + 1):
    # Simulate Bernoulli response
    y_k = 1 if np.random.uniform(0, 1) < theta_true else 0

    # Exact analytical conjugate update
    alpha_k += y_k
    beta_k += (1 - y_k)

    # Calculate point estimates
    bayes_est = alpha_k / (alpha_k + beta_k)

    # MAP estimate definition
    if alpha_k > 1 and beta_k > 1:
        map_est = (alpha_k - 1) / (alpha_k + beta_k - 2)
    else:
        map_est = bayes_est

    running_bayes.append(bayes_est)
    running_map.append(map_est)

# Plotting with Plotly
fig2 = go.Figure()

# True CTR reference line
fig2.add_hline(
    y=theta_true,
    line_dash="dash",
    line_color="red",
    annotation_text=f"True CTR (θ = {theta_true})"
)

# Posterior Mean Line
fig2.add_trace(go.Scatter(
    x=steps, y=running_bayes,
    mode='lines+markers',
    name='Posterior Mean (θ̂_Bayes)',
    line=dict(color='blue', width=2)
))

# MAP Line
fig2.add_trace(go.Scatter(
    x=steps, y=running_map,
    mode='lines+markers',
    name='MAP Estimate (θ̂_MAP)',
    line=dict(color='green', width=1.5, dash='dot')
))

fig2.update_layout(
    title="Sequential Beta-Binomial Estimation of CTR (θ) Over 100 Impressions",
    xaxis_title="Impression Step (k)",
    yaxis_title="Estimated CTR (θ̂)",
    template="plotly_white",
    hovermode="x unified"
)

fig2.show()

### Analysis & Interpretation

* **Distance Trajectory:** As impression count $k$ approaches $100$, both $\widehat{\theta}_{\text{Bayes}}^{(k)}$ and $\widehat{\theta}_{\text{MAP}}^{(k)}$ converge closely towards $\theta_{\text{true}} = 0.35$, with random variations decreasing over time.
* **Prior Influence vs. Sample Evidence:** At small $k$, the choice of prior influences the estimate. As $k$ grows, sample data $\sum y_i$ dominates the prior weights $\alpha_0, \beta_0$, causing the influence of the initial prior choice to wash out asymptotically (**Bernstein-von Mises theorem**).